In [10]:
# %% [markdown]
# # Main Analysis Notebook
# This notebook contains capacitance and image analysis for IDC sensors.

# %%
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

# %%
# --------- Add Analysis folder to Python path -----------
sys.path.append(r"C:\Users\goodn\OneDrive\Documents\GitHub\darts_idc_analysis\Analysis")

# ---------- Import helper modules ---------------
from Analysis import reads, adds

# %% [markdown]
# ## Capacitance Analysis Setup

# %%
print("Capacitance analysis")

# Paths
pristine_path = r"data/cv/cv_pristine"
exposed_path = r"data/cv/cv_exposed"
figures_path = r"figures 2"

# Create figures folder if it doesn't exist
os.makedirs(figures_path, exist_ok=True)

# %% [markdown]
# ## Load CSVs Function

# %%
def load_csv_folder(folder_path, condition_label):
    all_data = []
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)
            
            # Standardize column names
            df.columns = [col.split("(")[0].strip().replace(" ", "_") for col in df.columns]
            
            # Extract metadata
            parts = file.split("_")
            board_type = parts[1]
            board_number = parts[2]
            sensor_id = parts[3]  # U1, U2, etc.
            
            df["Board_Type"] = board_type
            df["Board_Number"] = board_number
            df["Sensor_ID"] = sensor_id
            df["Condition"] = condition_label
            df["Filename"] = file
            
            all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

# %% [markdown]
# ## Load Data

# %%
pristine_data = load_csv_folder(pristine_path, "pristine")
exposed_data = load_csv_folder(exposed_path, "exposed")
full_data = pd.concat([pristine_data, exposed_data], ignore_index=True)

# Sort the data
full_data = full_data.sort_values(["Board_Type","Sensor_ID","Condition","Voltage"])

# Interpolate per group
full_data["Capacitance"] = full_data.groupby(
    ["Board_Type","Sensor_ID","Condition"]
)["Capacitance"].transform(lambda x: x.interpolate())


# %% [markdown]
# ## Plot Capacitance vs Voltage

# %%
sns.set(style="whitegrid")

cond_colors = {
    "pristine": "blue",
    "exposed": "red"
}

for board_type, df_board in full_data.groupby("Board_Type"):
    plt.figure(figsize=(10,6))

    sns.lineplot(
        data=df_board,
        x="Voltage",
        y="Capacitance",
        hue="Condition",
        style="Sensor_ID",
        markers=True,
        dashes=False,
        palette=cond_colors,
        errorbar=None
    )
    plt.title(f"Capacitance vs Voltage for Board Type {board_type}")
    plt.xlabel("Voltage (V)")
    plt.ylabel("Capacitance (F)")

    plt.legend(title="Condition / Sensor", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    save_path = os.path.join(figures_path, f"Cap_vs_V_Board_{board_type}.png")
    plt.savefig(save_path, dpi=300)
    plt.show()
#this adds mean medium mode

# -------------------------
# Mean Capacitance vs Voltage per Board with annotation
# -------------------------
from scipy import stats

# Compute mean, median, mode per board, voltage, and condition
agg_stats = full_data.groupby(["Board_Type", "Voltage", "Condition"])["Capacitance"].agg(
    Mean="mean",
    Median="median",
    Mode=lambda x: stats.mode(x, keepdims=True).mode[0]
).reset_index()

for board_type in agg_stats["Board_Type"].unique():
    df_board = agg_stats[agg_stats["Board_Type"] == board_type]
    plt.figure(figsize=(10,6))

    for condition in df_board["Condition"].unique():
        df_cond = df_board[df_board["Condition"] == condition]
        base_color = "blue" if condition == "pristine" else "red"

        plt.plot(df_cond["Voltage"], df_cond["Mean"], color=base_color, linewidth=2, label=f"{condition} mean")
        plt.plot(df_cond["Voltage"], df_cond["Median"], color="purple", linewidth=2, linestyle="--", label=f"{condition} median")
        plt.plot(df_cond["Voltage"], df_cond["Mode"], color="green", linewidth=2, linestyle=":", label=f"{condition} mode")

    plt.title(f"Mean Capacitance vs Voltage - Board {board_type}")
    plt.xlabel("Voltage (V)")
    plt.ylabel("Capacitance (F)")

    # Add annotation inside plot
    plt.text(
        0.95, 0.95,
        "Blue/Red = mean\nPurple = median\nGreen = mode",
        horizontalalignment='right',
        verticalalignment='top',
        transform=plt.gca().transAxes,
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray')
    )

    plt.tight_layout()

    # Save figure
    save_path = os.path.join(figures_path, f"Board_{board_type}_mean_cap.png")
    plt.savefig(save_path, dpi=300)
    plt.show()

# %% [markdown]
# ## Compute Statistics (Mean, Median, Mode) per Sensor

# %%
def safe_mode(series):
    if len(series) == 0:
        return float("nan")
    return stats.mode(series, keepdims=True).mode[0]

stats_summary = full_data.groupby(["Board_Type","Condition","Sensor_ID"])["Capacitance"].agg(
    Mean="mean",
    Median="median",
    Mode=safe_mode
).reset_index()

stats_summary.to_csv(os.path.join(figures_path, "capacitance_stats_summary.csv"), index=False)


from scipy import stats

# Make figures folder if it doesn't exist
os.makedirs(figures_path, exist_ok=True)

# Function to safely compute mode
def safe_mode(series):
    if len(series) == 0:
        return float("nan")
    return stats.mode(series, keepdims=True).mode[0]

# Compute stats per Board, Sensor, and Condition
stats_summary = full_data.groupby(["Board_Type","Sensor_ID","Condition"])["Capacitance"].agg(
    Mean="mean",
    Median="median",
    Mode=safe_mode
).reset_index()

# Save stats CSV
stats_summary.to_csv(os.path.join(figures_path, "capacitance_stats_summary_per_sensor.csv"), index=False)

# Plot by sensor using stats
for board_type, df_board in full_data.groupby("Board_Type"):
    for sensor_id, df_sensor in df_board.groupby("Sensor_ID"):
        plt.figure(figsize=(8,5))
        sns.lineplot(
            data=df_sensor,
            x="Voltage",
            y="Capacitance",
            hue="Condition",
            palette={"pristine":"blue","exposed":"red"},
            markers=True,
            dashes=False,
            errorbar=None
        )
        plt.title(f"Board {board_type} - Sensor {sensor_id} Capacitance vs Voltage")
        plt.xlabel("Voltage (V)")
        plt.ylabel("Capacitance (F)")
        plt.legend(title="Condition")
        plt.tight_layout()

        # Save figure
        save_path = os.path.join(figures_path, f"Board_{board_type}_Sensor_{sensor_id}.png")
        plt.savefig(save_path, dpi=300)
        plt.show()


#----- one figure per board
from scipy import stats
from scipy import stats

# Compute mean, median, and mode per board, voltage, and condition
agg_stats = full_data.groupby(["Board_Type", "Voltage", "Condition"])["Capacitance"].agg(
    Mean="mean",
    Median="median",
    Mode=lambda x: stats.mode(x, keepdims=True).mode[0]
).reset_index()

for board_type, df_board in full_data.groupby("Board_Type"):
    plt.figure(figsize=(12,6))

    # Plot individual sensor CSVs
    for (sensor_id, condition, board_number), df_sensor in df_board.groupby(["Sensor_ID","Condition","Board_Number"]):
        color = "blue" if condition == "pristine" else "red"
        plt.plot(df_sensor["Voltage"], df_sensor["Capacitance"], color=color, alpha=0.5)

    # Overlay mean/median/mode with distinct colors/styles
    df_stats = agg_stats[agg_stats["Board_Type"] == board_type]
    for condition in df_stats["Condition"].unique():
        df_cond = df_stats[df_stats["Condition"] == condition]
        base_color = "blue" if condition == "pristine" else "red"
        plt.plot(df_cond["Voltage"], df_cond["Mean"], color=base_color, linewidth=2, label=f"{condition} mean")
        plt.plot(df_cond["Voltage"], df_cond["Median"], color="purple", linewidth=2, linestyle="--", label=f"{condition} median")
        plt.plot(df_cond["Voltage"], df_cond["Mode"], color="green", linewidth=2, linestyle=":", label=f"{condition} mode")

    plt.title(f"Capacitance vs Voltage - Board {board_type}")
    plt.xlabel("Voltage (V)")
    plt.ylabel("Capacitance (F)")

    # Optional: remove legend if plot is too busy
    # plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")

    plt.tight_layout()
    
    # Save figure
    save_path = os.path.join(figures_path, f"Board_{board_type}_all_sensors.png")
    plt.savefig(save_path, dpi=300)
    plt.show()



print("Plots and statistics saved in 'figures/' folder.")


Capacitance analysis
Plots and statistics saved in 'figures/' folder.
